# 03 Portfolio Backtest - MA Cross

Run an equal-sleeve MA Cross portfolio backtest and inspect contribution, risk, and equity behavior.


In [ ]:
# Cell 1 - Safe import path bootstrap

import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')


ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)


In [ ]:
# Cell 2 - Imports and notebook setup

from IPython.display import display

from core_python.strategies.ma_cross.research_utils import (
    configure_notebook,
    export_research_bundle,
    export_ctrader_validation_bundle,
    plot_portfolio_dashboard,
    run_portfolio_backtest,
    show_ftmo_check,
    show_note,
    show_portfolio_summary,
    show_run_config,
    show_strategy_summary,
    show_trade_explorer,
)

configure_notebook()
show_strategy_summary()


In [ ]:
# Cell 3 - Portfolio backtest configuration

RUN_CONFIG = {
    'symbols': ['US30', 'US100', 'GOLD', 'DE40', 'BTCUSD'],
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'tf': 'M30',
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'indicator_overrides': {},
    'strategy_overrides': {},
    'costs': {},
    'broker_profile': None,
    'export_report': False,
    'export_ctrader': False,
}
show_run_config('MA Cross Portfolio Configuration', RUN_CONFIG)


In [ ]:
# Cell 4 - Run portfolio backtest

portfolio = run_portfolio_backtest(**{k: v for k, v in RUN_CONFIG.items() if k not in {'export_report', 'export_ctrader'}})
show_portfolio_summary(portfolio)
plot_portfolio_dashboard(portfolio)
show_ftmo_check(portfolio.get('ftmo'))
show_trade_explorer(portfolio.get('trades'), title='Portfolio Recent Trades')


In [ ]:
# Cell 5 - Optional export

if RUN_CONFIG.get('export_report'):
    export_path = export_research_bundle(
        {
            'symbol_metrics': portfolio.get('symbol_metrics'),
            'trades': portfolio.get('trades'),
            'equity_frame': portfolio.get('equity_frame'),
            'combined_equity': portfolio.get('combined_equity'),
        },
        name='ma_cross_portfolio_backtest',
    )
    print('Exported:', export_path)
else:
    print("Export is disabled. Set RUN_CONFIG['export_report'] = True to save CSV files.")


In [ ]:
# Cell 6 - Optional cTrader validation export

if 'portfolio' not in globals():
    print('No portfolio result yet. Run Cell 4 first.')
elif RUN_CONFIG.get('export_ctrader'):
    out = export_ctrader_validation_bundle(
        portfolio,
        RUN_CONFIG,
        name=f"portfolio_{RUN_CONFIG['account_mode']}_{RUN_CONFIG['tf']}_backtest",
    )
    print('cTrader validation export:', out)
else:
    print("cTrader export is disabled. Set RUN_CONFIG['export_ctrader'] = True to save validation CSV files.")
